# 01 — Tier 1 Offline VAD Benchmark

This notebook provides the interactive test harness for **Tier 1 (Offline Bench)** evaluation.

### Tier 1 Workflow
1. Loads the audio corpus manifest via `corpus/manifest.py`.
2. Instantiates active VAD adapters (`ten_vad`, `firered_vad`, `webrtc_vad_mode0..3`) via `vad_adapters/adapter_registry.py`.
3. Feeds frame-by-frame 16kHz audio through the shared contract chunker.
4. Discards cold-start warm-up clips.
5. Resamples and scores decisions on the uniform 10ms scoring grid in `scoring/vad_scorer.py`.
6. Incrementally persists raw evaluation records to `storage/results_store.py`.

> **Boundary Rule**: This notebook contains zero scoring or VAD logic. It delegates execution entirely to `orchestrator.py`.

In [1]:
# Cell 1: Environment Setup & Working Directory Configuration
import sys
from pathlib import Path

# Ensure project root is present on sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root configured at: {project_root}")

Project root configured at: /workspaces/voice-bot-vad-harness


In [2]:
# Cell 2: Self-Contained Smoke Test Generator (2-3 placeholder clips, zero external audio needed)
import json
import numpy as np
import soundfile as sf
import yaml

smoke_test_dir = project_root / "tests" / "smoke_test_tier1"
smoke_audio_dir = smoke_test_dir / "audio"
smoke_results_dir = smoke_test_dir / "results"
smoke_audio_dir.mkdir(parents=True, exist_ok=True)
smoke_results_dir.mkdir(parents=True, exist_ok=True)

sample_rate = 16000
duration_s = 2.0
num_samples = int(sample_rate * duration_s)
t = np.linspace(0, duration_s, num_samples, endpoint=False, dtype=np.float32)

# Clip 1: Clean Speech (1.0s tone speech segment in the middle)
clean_audio = np.zeros(num_samples, dtype=np.float32)
clean_audio[8000:24000] = 0.5 * np.sin(2 * np.pi * 440.0 * t[:16000])
clean_path = smoke_audio_dir / "clip_01_clean.wav"
sf.write(str(clean_path), clean_audio, sample_rate)

# Clip 2: Speech + Stationary Noise (White noise background + tone speech)
noisy_audio = 0.05 * np.random.normal(0, 1, num_samples).astype(np.float32)
noisy_audio[8000:24000] += 0.4 * np.sin(2 * np.pi * 500.0 * t[:16000])
noisy_path = smoke_audio_dir / "clip_02_noisy.wav"
sf.write(str(noisy_path), noisy_audio, sample_rate)

# Clip 3: Silence-Only (Zero ground-truth speech segments)
silence_audio = np.zeros(num_samples, dtype=np.float32)
silence_path = smoke_audio_dir / "clip_03_silence.wav"
sf.write(str(silence_path), silence_audio, sample_rate)

# Write Smoke Test Manifest
manifest_entries = [
    {
        "clip_id": "smoke_clip_01_clean",
        "clip_path": str(clean_path),
        "category": "clean speech",
        "target_snr_db": None,
        "target_loudness_lufs": -20.0,
        "speech_segments": [[8000, 24000]],
        "calibration_marker_offset": 0,
        "is_held_out": False
    },
    {
        "clip_id": "smoke_clip_02_noisy",
        "clip_path": str(noisy_path),
        "category": "speech + stationary noise",
        "target_snr_db": 10.0,
        "target_loudness_lufs": -20.0,
        "speech_segments": [[8000, 24000]],
        "calibration_marker_offset": 0,
        "is_held_out": False
    },
    {
        "clip_id": "smoke_clip_03_silence",
        "clip_path": str(silence_path),
        "category": "silence-only",
        "target_snr_db": None,
        "target_loudness_lufs": None,
        "speech_segments": [],  # Silence-only explicit empty ground truth
        "calibration_marker_offset": 0,
        "is_held_out": False
    }
]

manifest_path = smoke_test_dir / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest_entries, f, indent=2)

# Write Smoke Test Run Config
smoke_config = {
    "run_id": "smoke_test_tier1_run",
    "tiers_to_run": ["tier1"],
    "active_adapters": ["webrtc_vad"],
    "webrtc_aggressiveness_modes": [0, 1, 2, 3],
    "corpus_manifest_path": str(manifest_path),
    "corpus_category_filter": [],
    "hardware_policy": "cpu_only",
    "warmup_clip_count": 0,  # 0 warmup so all 3 smoke clips are scored
    "scoring_grid_ms": 10,
    "concurrency_mode": "sequential",
    "include_held_out_slice": False,
    "results_destination": str(smoke_results_dir),
    "jitsi_room": "smoke-test-room",
    "bridge_url": "wss://smoke.test/feed"
}

config_path = smoke_test_dir / "run_config.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(smoke_config, f)

print(f"Smoke test environment prepared with 3 audio clips and manifest at: {manifest_path}")

Smoke test environment prepared with 3 audio clips and manifest at: /workspaces/voice-bot-vad-harness/tests/smoke_test_tier1/manifest.json


In [3]:
# Cell 3: The Single Substantive Call into orchestrator.py
from orchestrator import run_benchmark

# Execute Tier 1 benchmark through orchestrator
run_summary = run_benchmark(config_path=config_path)
print("\n--- Benchmark Execution Summary ---")
print(json.dumps(run_summary, indent=2))

2026-09-14 11:30:52,396 [INFO] [orchestrator] Loaded run configuration for run_id: 'smoke_test_tier1_run'


2026-09-14 11:30:52,397 [INFO] [corpus.manifest] Successfully loaded manifest from '/workspaces/voice-bot-vad-harness/tests/smoke_test_tier1/manifest.json' (3 clips, version_hash=e25a84ae252ce802)


2026-09-14 11:30:52,400 [INFO] [orchestrator] Loaded manifest '/workspaces/voice-bot-vad-harness/tests/smoke_test_tier1/manifest.json' (3 clips, hash=e25a84ae252ce802)


2026-09-14 11:30:52,402 [INFO] [storage.results_store] ResultsStore opened append-only store at '/workspaces/voice-bot-vad-harness/tests/smoke_test_tier1/results/raw_evaluations.jsonl'


2026-09-14 11:30:52,403 [INFO] [orchestrator] === Starting Tier 1 Offline Evaluation [Run ID: smoke_test_tier1_run] ===


...

2026-09-14 11:30:52,682 [INFO] [orchestrator] Tier 1 Completed for 'webrtc_vad_mode1': Evaluated 3 clips | Precision: 0.633 | Recall: 1.000 | F1: 0.775


2026-09-14 11:30:52,683 [INFO] [orchestrator] Tier 1 Completed for 'webrtc_vad_mode2': Evaluated 3 clips | Precision: 0.805 | Recall: 0.535 | F1: 0.643


2026-09-14 11:30:52,684 [INFO] [orchestrator] Tier 1 Completed for 'webrtc_vad_mode3': Evaluated 3 clips | Precision: 0.811 | Recall: 0.535 | F1: 0.645


2026-09-14 11:30:52,696 [INFO] [orchestrator] === Benchmark Run 'smoke_test_tier1_run' Completed Successfully ===



--- Benchmark Execution Summary ---
{
  "run_id": "smoke_test_tier1_run",
  "manifest_version": "e25a84ae252ce802",
  "tiers_executed": {
    "tier1": {
      "status": "success",
      "adapters_evaluated": [
        "webrtc_vad_mode0",
        "webrtc_vad_mode1",
        "webrtc_vad_mode2",
        "webrtc_vad_mode3"
      ]
    }
  }
}


In [4]:
# Cell 4: Display Persisted Evaluation Records from ResultsStore
from storage.results_store import ResultsStore

store = ResultsStore(smoke_results_dir)
records = list(store.read_raw_records(run_id="smoke_test_tier1_run"))

print(f"Total durable raw evaluation rows persisted: {len(records)}\n")

for r in records[:5]:
    print(f"[Adapter: {r['adapter_id']:<18}] [Clip: {r['clip_id']:<22}] [Type: {r['record_type']:<15}] "
          f"[Lib: {r['vad_library_version']}]")
    if r['record_type'] == 'scoring_clip':
        m = r['payload']['metrics']
        print(f"   --> Precision: {m['precision']:.3f} | Recall: {m['recall']:.3f} | F1: {m['f1']:.3f} | "
              f"FPR: {m['false_positive_rate']:.3f} | FNR: {m['false_negative_rate']:.3f}")

store.close()

2026-09-14 11:30:52,705 [INFO] [storage.results_store] ResultsStore opened append-only store at '/workspaces/voice-bot-vad-harness/tests/smoke_test_tier1/results/raw_evaluations.jsonl'


Total durable raw evaluation rows persisted: 31

[Adapter: audio_profiler    ] [Clip: smoke_clip_01_clean   ] [Type: audio_profile  ] [Lib: audio_profiler==1.0.0]
[Adapter: webrtc_vad_mode0  ] [Clip: smoke_clip_01_clean   ] [Type: scoring_clip   ] [Lib: webrtcvad-wheels==2.0.14]
   --> Precision: 0.862 | Recall: 1.000 | F1: 0.926 | FPR: 0.160 | FNR: 0.000
[Adapter: webrtc_vad_mode0  ] [Clip: smoke_clip_01_clean   ] [Type: tier1_algorithmic_latency] [Lib: webrtcvad-wheels==2.0.14]
[Adapter: webrtc_vad_mode1  ] [Clip: smoke_clip_01_clean   ] [Type: scoring_clip   ] [Lib: webrtcvad-wheels==2.0.14]
   --> Precision: 0.862 | Recall: 1.000 | F1: 0.926 | FPR: 0.160 | FNR: 0.000
[Adapter: webrtc_vad_mode1  ] [Clip: smoke_clip_01_clean   ] [Type: tier1_algorithmic_latency] [Lib: webrtcvad-wheels==2.0.14]
